# STEP 1: Overview of Script
# ---------------------------------------
# This script automates the assignment of roof constructions in ESP-r models.
#
# Workflow:
# 1. Load EPC dataset (contains BUILDING_REFERENCE_NUMBER and ROOF_CONS).
# 2. Load mapping file (ROOF_CONS → ESP-r assignment characters).
#    - Handles both single-letter and multi-line assignments (e.g. "0\nz").
# 3. Loop through each building folder in baseline_models.
# 4. Identify the model subfolder (e.g. Detached_2F, Flat, etc.).
# 5. Locate the cfg/roof.txt file.
# 6. Replace all instances of "ROOF" in roof.txt with the correct assignment.
# 7. Save the updated file.
#
# Notes:
# - Buildings without roof.txt are skipped.
# - Multi-line assignments preserve correct line breaks (critical for ESP-r).
# - Designed for use in Jupyter Notebook on macOS.

In [ ]:
# STEP 2: Import Libraries and Define Paths
# ---------------------------------------
import os
import pandas as pd

# File paths
epc_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_NPV_IRR_update.csv"
mapping_path = "/Users/jakegehrung/Desktop/data_raw/REFERENCE FILES/roof_ESP-r_assign.csv"
baseline_dir = "/Users/jakegehrung/Desktop/data_raw/baseline_models"

In [2]:
# STEP 3: Load EPC Dataset
# ---------------------------------------
epc_df = pd.read_csv(epc_path)

# Ensure correct columns exist
required_cols = ["BUILDING_REFERENCE_NUMBER", "ROOF_CONS"]
for col in required_cols:
    if col not in epc_df.columns:
        raise ValueError(f"Missing column: {col}")

# Convert building ID to string for matching folder names
epc_df["BUILDING_REFERENCE_NUMBER"] = epc_df["BUILDING_REFERENCE_NUMBER"].astype(str)

print(f"EPC dataset loaded: {len(epc_df)} records")

EPC dataset loaded: 117 records


In [3]:
# STEP 4 (FINAL FIX): Proper CSV Parsing with Multi-line Support
# ---------------------------------------
mapping_dict = {}

with open(mapping_path, "r") as f:
    content = f.read()

# Split into logical CSV rows (handle quoted multiline properly)
rows = []
current = ""
in_quotes = False

for char in content:
    if char == '"':
        in_quotes = not in_quotes

    if char == "\n" and not in_quotes:
        rows.append(current.strip())
        current = ""
    else:
        current += char

# Add last row
if current:
    rows.append(current.strip())

# Remove header
rows = rows[1:]

# Parse rows
for row in rows:
    if "," not in row:
        continue

    key, value = row.split(",", 1)

    key = key.strip()
    value = value.strip()

    # Remove surrounding quotes if present
    if value.startswith('"') and value.endswith('"'):
        value = value[1:-1]

    # Replace literal newlines correctly
    value = value.replace("\r", "").strip()

    mapping_dict[key] = value

# Debug check
print(f"Loaded {len(mapping_dict)} mappings:\n")
for k, v in list(mapping_dict.items())[:10]:
    print(f"{k} -> {repr(v)}")

Loaded 15 mappings:

insulated_roof_0.15 -> '0\nx'
insulated_roof_0.35 -> '0\nz'
insulated_roof_100mm -> 'o'
insulated_roof_12mm -> 'l'
insulated_roof_150mm -> 'p'
insulated_roof_200mm -> 'q'
insulated_roof_250mm -> 'r'
insulated_roof_270mm -> 's'
insulated_roof_300mm -> 't'
insulated_roof_350mm -> 'u'


In [4]:
#VALIDATION CHECK: Ensure all keys in mapping_dict exist in EPC dataset
print("\nTEST LOOKUPS:")
test_keys = [
    "insulated_roof_300mm",
    "insulated_roof_200mm",
    "insulated_roof_250mm",
    "uninsulated_roof"
]

for key in test_keys:
    print(f"{key} -> {repr(mapping_dict.get(key))}")


TEST LOOKUPS:
insulated_roof_300mm -> 't'
insulated_roof_200mm -> 'q'
insulated_roof_250mm -> 'r'
uninsulated_roof -> 'k'


In [ ]:
# STEP 5: Define Possible Model Folder Names
# ---------------------------------------
model_folder_names = [
    "SemiDetached_2F", "Detached_2F"
]

In [6]:
# STEP 6: Process Each Building
# ---------------------------------------
processed = 0
skipped = 0

for _, row in epc_df.iterrows():
    building_id = row["BUILDING_REFERENCE_NUMBER"]
    roof_cons = row["ROOF_CONS"]

    # Get assignment from mapping
    assignment = mapping_dict.get(roof_cons)

    if assignment is None:
        print(f"Skipping {building_id}: No mapping for {roof_cons}")
        skipped += 1
        continue

    building_path = os.path.join(baseline_dir, building_id)

    if not os.path.isdir(building_path):
        print(f"Skipping {building_id}: Folder not found")
        skipped += 1
        continue

    # Find model folder
    model_folder = None
    for name in model_folder_names:
        candidate = os.path.join(building_path, name)
        if os.path.isdir(candidate):
            model_folder = candidate
            break

    if model_folder is None:
        print(f"Skipping {building_id}: No model folder found")
        skipped += 1
        continue

    # Path to roof.txt
    roof_txt_path = os.path.join(model_folder, "cfg", "roof.txt")

    if not os.path.isfile(roof_txt_path):
        print(f"Skipping {building_id}: roof.txt not found")
        skipped += 1
        continue

    # Read file
    with open(roof_txt_path, "r") as f:
        content = f.read()

    # Replace placeholder
    updated_content = content.replace("ROOF", assignment)

    # Save file
    with open(roof_txt_path, "w") as f:
        f.write(updated_content)

    processed += 1

print("\n--- Processing Complete ---")
print(f"Processed: {processed}")
print(f"Skipped: {skipped}")

Skipping 1001829067: roof.txt not found
Skipping 1001951858: roof.txt not found
Skipping 1001829071: roof.txt not found
Skipping 1234761001: roof.txt not found
Skipping 1001991633: roof.txt not found
Skipping 1001829059: roof.txt not found
Skipping 1001063639: roof.txt not found
Skipping 1234761000: roof.txt not found
Skipping 1236594950: roof.txt not found
Skipping 1001664925: roof.txt not found
Skipping 1001906271: roof.txt not found
Skipping 1000238911: roof.txt not found
Skipping 1001829057: roof.txt not found
Skipping 1234760999: roof.txt not found
Skipping 1002143357: roof.txt not found
Skipping 1001951854: roof.txt not found
Skipping 1001829069: roof.txt not found
Skipping 1002313096: roof.txt not found
Skipping 1002143351: roof.txt not found
Skipping 1001870854: roof.txt not found
Skipping 1001870864: roof.txt not found
Skipping 1002143293: roof.txt not found
Skipping 1002143352: roof.txt not found
Skipping 1002143353: roof.txt not found
Skipping 1234647955: No model folder fou

In [7]:
# STEP 7: Optional Validation (Check a Few Files)
# ---------------------------------------
# Randomly print a few updated files to verify correctness
import random

sample_ids = random.sample(list(epc_df["BUILDING_REFERENCE_NUMBER"]), 5)

for building_id in sample_ids:
    building_path = os.path.join(baseline_dir, building_id)

    for name in model_folder_names:
        model_folder = os.path.join(building_path, name)
        roof_txt_path = os.path.join(model_folder, "cfg", "roof.txt")

        if os.path.isfile(roof_txt_path):
            print(f"\n--- {building_id} ---")
            with open(roof_txt_path, "r") as f:
                print(f.read())
            break


--- 1001664929 ---
m
c
a
c
f
*
b
f
r
e
-
y
b
-
!

y
-
y
-
-
!


-
-

